# Tumor / Non-tumor Mask Generation

HER2 YOLOv11 모델 prediction point를 이용해 patch image 폴더 전체에 대해 tumor / non-tumor pseudo-mask를 생성합니다.

- `class0`-`class3`: tumor-cell 후보
- `other`: non-tumor 후보
- background: near-white background 또는 cell-density support 밖

저장되는 mask 값:

| value | region |
|---:|---|
| 0 | background |
| 1 | tumor |
| 2 | non-tumor |


In [1]:
from pathlib import Path

# Input / output paths
IMAGE_DIR = Path('../../data/precise_BC_cell_scoring/her2/patch_images')
LABEL_DIR = Path('../../data/precise_BC_cell_scoring/her2/labels')
CHECKPOINT = Path('../../model/precise_BC_cell_scoring/her2_yolov11/best_model.pt')
OUT_DIR = Path('../../data/precise_BC_cell_scoring/her2/tumor_non_tumor_masks')

# Point source: 'model' uses YOLO prediction, 'label' uses existing json labels.
SOURCE = 'model'

# Set to None for the whole folder. Use a small number for a quick test.
MAX_IMAGES = None

# Model / mask parameters
INPUT_SIZE = 512
CONF_THRESHOLD = 0.5
NMS_IOU = 0.45
SIGMA = 24.0
ACTIVE_THRESHOLD = 0.03
TUMOR_THRESHOLD = 0.5

# Save options
SAVE_LABEL_MASK = True
SAVE_COLOR_MASK = True
SAVE_OVERLAY = True

# Skip patches where generated background occupies more than this ratio.
MAX_BACKGROUND_RATIO = 0.5

MASK_DIR = OUT_DIR / 'label_masks'
COLOR_DIR = OUT_DIR / 'color_masks'
OVERLAY_DIR = OUT_DIR / 'overlays'
for d in [MASK_DIR, COLOR_DIR, OVERLAY_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('IMAGE_DIR:', IMAGE_DIR.resolve())
print('OUT_DIR  :', OUT_DIR.resolve())


IMAGE_DIR: /home/user/urbandatalab/YSLee/data/precise_BC_cell_scoring/her2/patch_images
OUT_DIR  : /home/user/urbandatalab/YSLee/data/precise_BC_cell_scoring/her2/tumor_non_tumor_masks


In [2]:
import json
import os
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from nets import nn
from utils import util
from scripts.make_tumor_region_examples import (
    CLASS_NAMES,
    label_points,
    load_image,
    load_model,
    make_density,
    make_tissue_support,
    overlay_regions,
    predict_points,
    smooth_filled_regions,
)

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = lambda x, **kwargs: x

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


device: cuda


/home/user/anaconda3/envs/urban/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def build_region_masks(image, points):
    tumor_density = make_density(points, [0, 1, 2, 3], image.shape[:2], SIGMA)
    other_density = make_density(points, [4], image.shape[:2], SIGMA)
    total_density = tumor_density + other_density
    tumor_score = tumor_density / (total_density + 1e-6)

    support = make_tissue_support(image, total_density, ACTIVE_THRESHOLD)
    tumor_region = smooth_filled_regions((tumor_score >= TUMOR_THRESHOLD) & support, support)

    tumor_mask = tumor_region.astype(np.uint8) * 255
    non_tumor_mask = (support & (~tumor_region)).astype(np.uint8) * 255

    label_mask = np.zeros(image.shape[:2], dtype=np.uint8)
    label_mask[tumor_mask > 0] = 1
    label_mask[non_tumor_mask > 0] = 2

    return {
        'tumor_density': tumor_density,
        'other_density': other_density,
        'tumor_score': tumor_score,
        'tumor_mask': tumor_mask,
        'non_tumor_mask': non_tumor_mask,
        'label_mask': label_mask,
    }


def label_to_color(label_mask):
    color = np.zeros((*label_mask.shape, 3), dtype=np.uint8)
    color[label_mask == 1] = (255, 40, 40)     # tumor: red
    color[label_mask == 2] = (30, 180, 255)    # non-tumor: blue/cyan
    return color


def save_patch_outputs(image_path, image, masks):
    stem = image_path.stem
    label_mask = masks['label_mask']
    tumor_mask = masks['tumor_mask']
    non_tumor_mask = masks['non_tumor_mask']

    if SAVE_LABEL_MASK:
        cv2.imwrite(str(MASK_DIR / f'{stem}_mask.png'), label_mask)

    if SAVE_COLOR_MASK:
        color = label_to_color(label_mask)
        cv2.imwrite(str(COLOR_DIR / f'{stem}_color_mask.png'), cv2.cvtColor(color, cv2.COLOR_RGB2BGR))

    if SAVE_OVERLAY:
        overlay = overlay_regions(image, tumor_mask, non_tumor_mask)
        cv2.imwrite(str(OVERLAY_DIR / f'{stem}_overlay.png'), cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))


In [4]:
image_paths = sorted(IMAGE_DIR.glob('*.png'))
if MAX_IMAGES is not None:
    image_paths = image_paths[:MAX_IMAGES]

print('num images:', len(image_paths))
assert len(image_paths) > 0, f'No PNG images found in {IMAGE_DIR}'


num images: 14401


In [5]:
model = None
if SOURCE == 'model':
    model = load_model(CHECKPOINT, len(CLASS_NAMES), device)
    print('loaded checkpoint:', CHECKPOINT)
elif SOURCE == 'label':
    print('using json labels as point source')
else:
    raise ValueError("SOURCE must be 'model' or 'label'")


loaded checkpoint: ../../model/precise_BC_cell_scoring/her2_yolov11/best_model.pt


In [6]:
summary = []

for image_path in tqdm(image_paths, desc='Generating masks'):
    image, tensor = load_image(image_path, INPUT_SIZE)

    if SOURCE == 'model':
        points = predict_points(model, tensor, device, CONF_THRESHOLD, NMS_IOU)
    else:
        points = label_points(LABEL_DIR / f'{image_path.stem}.json', INPUT_SIZE)

    masks = build_region_masks(image, points)
    label_mask = masks['label_mask']
    tumor_px = int((label_mask == 1).sum())
    non_tumor_px = int((label_mask == 2).sum())
    bg_px = int((label_mask == 0).sum())
    total_px = int(label_mask.size)
    background_ratio = bg_px / total_px
    tumor_points = int(np.isin(points[:, 2].astype(np.int32), [0, 1, 2, 3]).sum()) if len(points) else 0
    other_points = int((points[:, 2].astype(np.int32) == 4).sum()) if len(points) else 0
    skipped = background_ratio > MAX_BACKGROUND_RATIO

    if not skipped:
        save_patch_outputs(image_path, image, masks)

    summary.append({
        'image': image_path.name,
        'saved': bool(not skipped),
        'skip_reason': 'background_ratio_gt_threshold' if skipped else '',
        'points': int(len(points)),
        'tumor_points': tumor_points,
        'other_points': other_points,
        'tumor_px': tumor_px,
        'non_tumor_px': non_tumor_px,
        'background_px': bg_px,
        'background_ratio': float(background_ratio),
    })

summary_path = OUT_DIR / 'mask_generation_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print('saved summary:', summary_path)
print('saved patches:', sum(1 for row in summary if row['saved']))
print('skipped patches:', sum(1 for row in summary if not row['saved']))
print('saved label masks:', MASK_DIR)
print('saved color masks:', COLOR_DIR)
print('saved overlays:', OVERLAY_DIR)


Generating masks:   0%|          | 0/14401 [00:00<?, ?it/s]/home/user/anaconda3/envs/urban/lib/python3.12/site-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4317.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
Generating masks: 100%|██████████| 14401/14401 [15:03<00:00, 15.94it/s]

saved summary: ../../data/precise_BC_cell_scoring/her2/tumor_non_tumor_masks/mask_generation_summary.json
saved patches: 10089
skipped patches: 4312
saved label masks: ../../data/precise_BC_cell_scoring/her2/tumor_non_tumor_masks/label_masks
saved color masks: ../../data/precise_BC_cell_scoring/her2/tumor_non_tumor_masks/color_masks
saved overlays: ../../data/precise_BC_cell_scoring/her2/tumor_non_tumor_masks/overlays


In [7]:
# Preview a few generated overlays.
preview_paths = sorted(OVERLAY_DIR.glob('*_overlay.png'))[:6]

cols = 3
rows = int(np.ceil(len(preview_paths) / cols)) if preview_paths else 1
fig, axes = plt.subplots(rows, cols, figsize=(12, 4 * rows))
axes = np.array(axes).reshape(-1)

for ax, path in zip(axes, preview_paths):
    img = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(path.name, fontsize=8)
    ax.axis('off')

for ax in axes[len(preview_paths):]:
    ax.axis('off')

plt.tight_layout()
